In [ ]:
!pip install accelerate datasets scikit-learn -q

import os
import sys
import shutil

KAGGLE_INPUT_DIR = "/kaggle/input/nexus_absa_kaggle_input"

if not os.path.exists("/kaggle/working/modules"):
    shutil.copytree(os.path.join(KAGGLE_INPUT_DIR, "modules"), "/kaggle/working/modules")

sys.path.append("/kaggle/working")

print("Modules loaded and environment is ready!")

In [ ]:
import gc
import time
import json
import torch
import numpy as np
from tqdm.auto import tqdm
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from modules.config import Config
from modules.data_loader import get_dataloaders
from modules.knowledge_modules import SymbolicModuleSenticNet, ConceptNetModule
from modules.model_architectures import NeuroSymbolicEncoder, NeuroSymbolicCausalLM, apply_tada
from modules.xai_engine import XAIEngine

os.environ["TOKENIZERS_PARALLELISM"] = "false"

accelerator = Accelerator()

accelerator.print(f"Distributed Training Initialized.")
accelerator.print(f"Number of GPUs being used: {accelerator.num_processes}")
accelerator.print(f"Device: {accelerator.device}")

In [ ]:
def run_training():
    results = []
    xai_engine = XAIEngine(accelerator.device)
    
    for scenario in Config.SCENARIOS:
        sid, model_path, arch_type, strategy, k_source, ds_name = scenario
        scenario_id_str = f"Scenario {sid}: {arch_type.upper()} | {strategy} | {ds_name}"
        
        accelerator.print(f"\n{'='*60}\nSTARTING {scenario_id_str}\n{'='*60}")

        if k_source == 'senticnet':
            sym_module = SymbolicModuleSenticNet(Config.SENTICNET_CACHE_PATH)
        else:
            sym_module = ConceptNetModule(Config.CONCEPTNET_CACHE_PATH)

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, trust_remote_code=True)
            tokenizer.padding_side = 'right'
            if tokenizer.pad_token is None: 
                tokenizer.pad_token = tokenizer.eos_token
        except Exception as e:
            accelerator.print(f"Tokenizer error: {e}"); continue

        train_loader, val_loader, current_max_len = get_dataloaders(Config, tokenizer, sym_module, k_source, ds_name)

        try:
            if arch_type in ['roberta', 'deberta']: 
                model = NeuroSymbolicEncoder(model_path)
            else: 
                model = NeuroSymbolicCausalLM(model_path)
                
            backbone = model.backbone
            if len(tokenizer) > backbone.get_input_embeddings().num_embeddings:
                backbone.resize_token_embeddings(len(tokenizer))
                
            apply_tada(model, arch_type, strategy)
        except Exception as e: 
            accelerator.print(f"Model error: {e}"); continue

        optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LEARNING_RATE)
        total_steps = len(train_loader) * Config.EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
        loss_fn = torch.nn.CrossEntropyLoss()

        model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
            model, optimizer, train_loader, val_loader, scheduler
        )

        best_f1 = 0.0
        best_metrics = {}
        start_time = time.time()
        
        # ---------------- TRAINING LOOP ----------------
        for epoch in range(Config.EPOCHS):
            model.train()
            train_loss = 0.0
            
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", disable=not accelerator.is_local_main_process)
            
            for batch in progress_bar:
                with accelerator.accumulate(model):
                    kwargs = {}
                    if 'token_type_ids' in batch: 
                        kwargs['token_type_ids'] = batch['token_type_ids']

                    outputs = model(batch['input_ids'], batch['attention_mask'], batch['symbolic_features'], **kwargs)
                    loss = loss_fn(outputs, batch['labels'])
                    
                    accelerator.backward(loss)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    
                    train_loss += loss.item()

            # ---------------- VALIDATION LOOP ----------------
            model.eval()
            all_preds, all_labels = [], []
            val_texts, val_aspects = [], []
            
            with torch.no_grad():
                for batch in val_loader:
                    kwargs = {}
                    if 'token_type_ids' in batch: kwargs['token_type_ids'] = batch['token_type_ids']
                    
                    outputs = model(batch['input_ids'], batch['attention_mask'], batch['symbolic_features'], **kwargs)
                    preds = torch.argmax(outputs, dim=1)
                    
                    gathered_preds, gathered_labels = accelerator.gather_for_metrics((preds, batch['labels']))
                    
                    all_preds.extend(gathered_preds.cpu().numpy())
                    all_labels.extend(gathered_labels.cpu().numpy())
                    
                    if accelerator.is_main_process:
                        val_texts.extend(batch['raw_text'])
                        val_aspects.extend(batch['aspect'])

            if accelerator.is_main_process:
                acc = accuracy_score(all_labels, all_preds)
                p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
                
                avg_loss = train_loss / len(train_loader)
                accelerator.print(f"Epoch {epoch+1}: Loss={avg_loss:.4f} | Macro-F1={f1:.4f} | Acc={acc:.4f}")

                if f1 > best_f1:
                    best_f1 = f1
                    best_metrics = {"accuracy": acc, "precision": p, "recall": r, "f1": f1}
                    accelerator.print(f"--> New Best Model Found (Macro-F1: {best_f1:.4f})")
                    
                    best_val_texts = val_texts
                    best_val_aspects = val_aspects
                    best_all_labels = all_labels
                    best_all_preds = all_preds

        # ---------------- XAI EVALUATION (End of Scenario) ----------------
        accelerator.wait_for_everyone()
        
        if accelerator.is_main_process and best_f1 > 0:
            accelerator.print(f"Calculating XAI Metrics for Scenario {sid}...")
            
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.eval()
            
            xai_res = {"sufficiency": [], "infidelity": []}
            
            sample_limit = min(200, len(best_val_texts))
            
            for idx in tqdm(range(sample_limit), desc="XAI Calc"):
                txt = best_val_texts[idx]
                asp = best_val_aspects[idx]
                lbl = best_all_labels[idx]
                prd = best_all_preds[idx]
                
                if k_source == 'senticnet': 
                    _, kws = sym_module.get_text_polarity(txt)
                else: 
                    kws = sym_module.get_keywords_from_text(txt)
                    
                metrics = xai_engine.calculate_metrics(unwrapped_model, tokenizer, txt, asp, lbl, prd, kws, sym_module, current_max_len)
                xai_res["sufficiency"].append(metrics["sufficiency"])
                xai_res["infidelity"].append(metrics["infidelity"])
                
            best_metrics["avg_sufficiency"] = np.mean(xai_res["sufficiency"])
            best_metrics["avg_infidelity"] = np.mean(xai_res["infidelity"])
            
            scenario_result = {
                "id": sid, "model": arch_type, "strategy": strategy, "dataset": ds_name,
                "metrics": best_metrics, "training_time_seconds": time.time() - start_time
            }
            results.append(scenario_result)
            
            with open(Config.RESULTS_FILE, "w") as f: 
                json.dump(results, f, indent=4)
            accelerator.print(f"Results saved to {Config.RESULTS_FILE}")

        del model, optimizer, scheduler, train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()

    if accelerator.is_main_process:
        accelerator.print("\n🎉 All scenarios completed successfully!")

run_training()